In [ ]:
# --- Importaciones ---------------------------------------------------
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')
from pathlib import Path
import sklearn, xgboost

print(f'sklearn:  {sklearn.__version__}')
print(f'xgboost:  {xgboost.__version__}')
print(f'numpy:    {np.__version__}')
print(f'pandas:   {pd.__version__}')

In [ ]:
# --- Configurar credenciales de Kaggle -----------------------------------
import os
import json
from google.colab import userdata

# Crea el directorio .kaggle si no existe
if not os.path.exists('/root/.kaggle'):
    os.makedirs('/root/.kaggle')

# Intenta obtener las credenciales de los secretos de Colab
try:
    kaggle_username = userdata.get('KAGGLE_USERNAME')
    kaggle_key = userdata.get('KAGGLE_KEY')

    if kaggle_username and kaggle_key:
        # Crea el archivo kaggle.json
        with open('/root/.kaggle/kaggle.json', 'w') as f:
            json.dump({"username": kaggle_username, "key": kaggle_key}, f)
        os.chmod('/root/.kaggle/kaggle.json', 600) # Establece permisos
        print('Las credenciales de Kaggle se han configurado correctamente.')
    else:
        raise ValueError('No se ha encontrado KAGGLE_USERNAME ni KAGGLE_KEY.')
except Exception as e:
    print(f'Advertencia: No se han podido recuperar las credenciales de Kaggle: {e}')
    print('Asegúrate de que KAGGLE_USERNAME y KAGGLE_KEY estén configurados en los secretos de Colab o introdúcelos manualmente.')
    # Recurrir a la introducción manual (menos seguro para la clave de API)
    try:
        kaggle_username = input('Introduce tu nombre de usuario de Kaggle: ')
        kaggle_key = input('Introduce tu clave de API de Kaggle: ')
        with open('/root/.kaggle/kaggle.json', 'w') as f:
            json.dump({"username": kaggle_username, "key": kaggle_key}, f)
        os.chmod('/root/.kaggle/kaggle.json', 600)
        print('Las credenciales de Kaggle se han configurado correctamente tras introducirlas manualmente.')
    except Exception as e:
        print(f'Error al configurar manualmente las credenciales de Kaggle: {e}')
        print('Asegúrate de que tengas un archivo kaggle.json válido o de que las variables de entorno estén configuradas.')

# Comprobar la configuración
!kaggle config view
!kaggle competitions list -s house-prices

In [ ]:
# --- Carga del dataset House Prices ---------------------------------
# Opción 1: desde Kaggle API (requiere cuenta en Kaggle y credenciales válidas)
!pip install kaggle --quiet
!kaggle competitions download -c house-prices-advanced-regression-techniques
!unzip -o house-prices-advanced-regression-techniques.zip

# Opción 2 (fallback): URLs públicas con el dataset Ames Housing (si la descarga de Kaggle falla)
# Nuevas URLs verificadas:
url_train = ('https://raw.githubusercontent.com/datasets/house-prices-advanced-regression-techniques/main/train.csv')
url_test  = ('https://raw.githubusercontent.com/datasets/house-prices-advanced-regression-techniques/main/test.csv')

try:
    # Intentar cargar desde archivos locales (si usaste Kaggle API y tuvo éxito)
    df_train = pd.read_csv('train.csv')
    df_test   = pd.read_csv('test.csv')
    print('Datos cargados desde archivos locales de Kaggle.')
except FileNotFoundError:
    print('Los archivos de Kaggle no se encontraron localmente. Intentando cargar desde URLs públicas de respaldo...')
    try:
        df_train = pd.read_csv(url_train)
        df_test  = pd.read_csv(url_test)
        print('Datos cargados desde URLs públicas de respaldo.')
    except Exception as e:
        print(f'Error al cargar desde URLs públicas: {e}')
        print('Asegúrate de que las URLs sean válidas o verifica tus credenciales de Kaggle.')

# Verifica si los dataframes se cargaron antes de intentar acceder a .shape
if 'df_train' in locals() and 'df_test' in locals():
    print(f'Train: {df_train.shape} | Test: {df_test.shape}')
else:
    print('Los dataframes df_train y df_test no se pudieron cargar.')

In [ ]:
# --- Inspección rápida del dataset -----------------------------------
print(df_train.head(3).to_string())
print(df_train.dtypes.value_counts())
# object    43   <- variables categóricas (texto)
# int64     35   <- variables numéricas enteras
# float64    3   <- variables numéricas decimales

# Columnas con más nulos (top 10)
nulos = df_train.isnull().mean().sort_values(ascending=False)
print(nulos[nulos > 0].head(10).round(3))

In [ ]:
# --- Distribución del precio y su transformación log ----------------
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

ax1.hist(df_train['SalePrice'], bins=50, color='steelblue', edgecolor='w')
ax1.set_title('Distribución original de SalePrice')
ax1.set_xlabel('Precio ($)')

ax2.hist(np.log1p(df_train['SalePrice']), bins=50, color='steelblue', edgecolor='w')
ax2.set_title('Distribución log(1+SalePrice)')
ax2.set_xlabel('log(1 + Precio)')

plt.suptitle('Efecto de la transformación logarítmica')
plt.tight_layout()
plt.show()

# Aplicar log1p al target
y_train = np.log1p(df_train['SalePrice'])
print(f'Target: media={y_train.mean():.3f} std={y_train.std():.3f}')

In [ ]:
# --- Correlación de variables numéricas con log(precio) --------------
df_num = df_train.select_dtypes(include=[np.number]).copy()
df_num['log_price'] = y_train

correlaciones = (
    df_num.corr()['log_price']
    .drop('log_price')
    .abs()
    .sort_values(ascending=False)
)
print('Top 10 features más correlacionadas con log(precio):')
print(correlaciones.head(10).round(3))


In [ ]:
# --- Combinar train y test para preprocesar juntos ------------------
# (importante: NO usamos y_test, solo las X)
n_train = len(df_train)
df_all = pd.concat([
    df_train.drop(columns=['SalePrice']),
    df_test
], axis=0).reset_index(drop=True)

# --- Eliminar columnas con >80% nulos --------------------------------
cols_drop = ['PoolQC', 'MiscFeature', 'Alley', 'Fence']
df_all = df_all.drop(columns=cols_drop)

# --- Feature engineering: superficie total --------------------------
# Decisión: combinar superficies porque el modelo las ve separadas
# y su suma captura mejor el concepto de 'tamaño total de la casa'
df_all['TotalSF'] = (
    df_all['TotalBsmtSF'].fillna(0) +
    df_all['1stFlrSF'].fillna(0) +
    df_all['2ndFlrSF'].fillna(0)
)

# Antigüedad al momento de venta
df_all['Age'] = df_all['YrSold'] - df_all['YearBuilt']

# Indicador: ¿fue remodelado?
df_all['Remodeled'] = (
    df_all['YearRemodAdd'] != df_all['YearBuilt']
).astype(int)

print(f'Shape tras engineering: {df_all.shape}')

In [ ]:
# --- Identificar tipos de columnas ----------------------------------
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OrdinalEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

# Separar train y test del dataframe combinado
X_train_raw = df_all.iloc[:n_train].copy()
X_test_raw  = df_all.iloc[n_train:].copy()

cols_num = X_train_raw.select_dtypes(include=[np.number]).columns.tolist()
cols_cat = X_train_raw.select_dtypes(include=['object']).columns.tolist()

print(f'Features numéricas: {len(cols_num)}')
print(f'Features categóricas: {len(cols_cat)}')

# Pipeline numérico: imputar mediana + escalar
pipe_num = Pipeline([
    ('imp', SimpleImputer(strategy='median')),
    ('sc',  StandardScaler()),
])

# Pipeline categórico: imputar moda + encoding ordinal
# Usamos OrdinalEncoder (no OneHot) porque XGBoost maneja bien
# variables ordinales y evitamos la explosión de dimensiones
pipe_cat = Pipeline([
    ('imp', SimpleImputer(strategy='most_frequent')),
    ('enc', OrdinalEncoder(
        handle_unknown='use_encoded_value', unknown_value=-1
    )),
])

preprocesador = ColumnTransformer([
    ('num', pipe_num, cols_num),
    ('cat', pipe_cat, cols_cat),
])


In [ ]:
# --- Setup de evaluación ---------------------------------------------
from sklearn.model_selection import cross_val_score, KFold
from sklearn.linear_model import Ridge
from sklearn.ensemble import RandomForestRegressor
from xgboost import XGBRegressor
from sklearn.metrics import mean_squared_error

semilla = 42
kf = KFold(n_splits=5, shuffle=True, random_state=semilla)

def evaluar(nombre, modelo, X, y, cv):
    """CV RMSE sobre log(precio)."""
    scores = cross_val_score(
        modelo, X, y,
        cv=cv,
        scoring='neg_root_mean_squared_error',
        n_jobs=-1,
    )
    rmse_cv = -scores.mean()
    std_cv  = scores.std()
    print(f'{nombre:<30} RMSE CV: {rmse_cv:.4f} ± {std_cv:.4f}')
    return rmse_cv

In [ ]:
# --- Ridge: modelo lineal regularizado (baseline) -------------------
# Decisión: elegimos Ridge (no Lasso) porque sospechamos que muchas
# features son útiles, no solo un subconjunto pequeño.
ridge_pipe = Pipeline([
    ('prep', preprocesador),
    ('clf', Ridge(alpha=10.0)),
])
rmse_ridge = evaluar('Ridge (alpha=10)', ridge_pipe, X_train_raw, y_train, kf)

In [ ]:
# --- Random Forest ---------------------------------------------------
# Decisión: no escala, por eso no usamos pipe_num con StandardScaler
# para RF. Sin embargo, mantenemos el pipeline completo para coherencia
# (el escalado no perjudica a RF aunque tampoco ayuda).
rf_pipe = Pipeline([
    ('prep', preprocesador),
    ('clf', RandomForestRegressor(
        n_estimators=200,
        random_state=semilla,
        n_jobs=-1,
    )),
])
rmse_rf = evaluar('RandomForest (200 est.)', rf_pipe, X_train_raw, y_train, kf)

In [ ]:
# --- XGBoost con parámetros iniciales --------------------------------
# Decisión: empezamos con learning_rate=0.05 (conservador) y
# n_estimators=500. Usaremos early stopping en la Fase 5 para
# encontrar el n_estimators óptimo.
xgb_pipe = Pipeline([
    ('prep', preprocesador),
    ('clf', XGBRegressor(
        n_estimators=500,
        learning_rate=0.05,
        max_depth=4,
        subsample=0.8,
        colsample_bytree=0.8,
        random_state=semilla,
        n_jobs=-1,
        verbosity=0,
    )),
])
rmse_xgb = evaluar('XGBoost (500 est.)', xgb_pipe, X_train_raw, y_train, kf)

In [ ]:
# --- RandomizedSearchCV sobre XGBoost -------------------------------
from sklearn.model_selection import RandomizedSearchCV
import scipy.stats as stats

param_dist = {
    'clf__n_estimators':     [300, 500, 700],
    'clf__learning_rate':    stats.loguniform(0.01, 0.1),
    'clf__max_depth':        [3, 4, 5, 6],
    'clf__subsample':        stats.uniform(0.6, 0.4),
    'clf__colsample_bytree': stats.uniform(0.6, 0.4),
    'clf__reg_alpha':        stats.loguniform(0.001, 1.0),
    'clf__reg_lambda':       stats.loguniform(0.1, 10.0),
    'clf__min_child_weight': [1, 3, 5],
}

rs = RandomizedSearchCV(
    xgb_pipe,
    param_distributions=param_dist,
    n_iter=30,
    cv=kf,
    scoring='neg_root_mean_squared_error',
    n_jobs=-1,
    random_state=semilla,
    verbose=1,
)

rs.fit(X_train_raw, y_train)

print(f'Mejor RMSE CV: {-rs.best_score_:.4f}')
print(f'Mejores params: {rs.best_params_}')


In [ ]:
# --- Entrenar el modelo final sobre todos los datos de train --------
modelo_final = rs.best_estimator_
modelo_final.fit(X_train_raw, y_train)

# --- Generar predicciones sobre el test set -------------------------
# Las predicciones están en log(1+precio); hay que invertir con expm1
log_pred = modelo_final.predict(X_test_raw)
predicciones = np.expm1(log_pred)

print(f'Predicciones: min={predicciones.min():.0f}',f'max={predicciones.max():.0f}',f'media={predicciones.mean():.0f}')

In [ ]:
# --- Generar submission.csv ------------------------------------------
submission = pd.DataFrame({
    'Id':        df_test['Id'],
    'SalePrice': predicciones,
})

ruta_submission = Path('submission.csv')
try:
    submission.to_csv(ruta_submission, index=False)
    print(f'Archivo guardado: {ruta_submission}')
    print(submission.head())
except Exception as e:
    print(f'Error al guardar: {e}')